# Week 6 — Notebook 1: Binary Classification Framing

**Difficulty:** ⭐⭐ | **Estimated Time:** 2 hours  
**Theme:** Spam Email Detection (spam = 1, not-spam = 0)

---

## Why This Matters

You have already learned how regression predicts *how much* — house prices, temperatures, sales figures.  
But many real decisions are categorical: **yes or no, spam or not, malignant or benign, fraud or legitimate.**

Classification is arguably the most commercially impactful branch of supervised ML.  
Every time Gmail silently moves a scam into your Spam folder, a classification model made a binary decision.  
This notebook teaches you how to frame, measure, and think about that decision correctly.

---

## The Analogy: A Spam Filter is a Judge in a Courtroom

Imagine every incoming email is a **defendant** standing before a judge.  
The judge examines **evidence** (email length, number of links, suspicious words) and delivers a verdict:

- **Guilty → Spam (1):** sent to the Spam folder
- **Innocent → Not Spam (0):** delivered to your inbox

The judge does not say "this email *is* spam" with certainty.  
Instead, the judge computes an internal **confidence score** (a probability) and crosses a threshold to reach a verdict.

Change the threshold and you change the character of the justice system:
- **Low threshold** → convict more liberally → catch more spam but also flag legitimate emails *(high recall, low precision)*
- **High threshold** → convict only when very sure → miss some spam but never wrongly convict *(high precision, low recall)*

Sound familiar? It is exactly the precision–recall tradeoff you will see throughout this notebook.

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain what a **model output probability** P(y=1|x) means
2. Adjust the **decision threshold** and understand its effect on precision and recall
3. Interpret a **confusion matrix** and compute precision, recall, and F1
4. Read an **ROC curve** and understand AUC
5. Explain the **accuracy paradox** and why it matters for imbalanced data
6. Visualise a **decision boundary** in 2D feature space

## Section 1 — From Regression to Classification

| Task | Output | Example |
|------|--------|---------|
| Regression | Continuous real number | House price = £342,000 |
| Binary Classification | One of two classes + probability | P(spam) = 0.87 → spam |
| Multi-class Classification | One of K classes | Digit = {0,1,…,9} |

### Plain English Definition

> A **classifier** learns a function f(x) → [0,1] that outputs the probability that input x belongs to class 1.  
> We then apply a **threshold t** (commonly 0.5) to convert that probability into a hard class label:
> - If f(x) ≥ t → predict class 1 (spam)
> - If f(x) < t → predict class 0 (not-spam)

### The Key Insight

The model outputs a **probability**, not a class.  
The threshold converts probability → class.  
You can always change the threshold *after* training without retraining the model.

In [ ]:
# ── Cell 1: Imports and global settings ──────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score,
    f1_score, roc_curve, auc, accuracy_score
)
from sklearn.preprocessing import StandardScaler

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Plot aesthetics
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

print("Libraries loaded successfully.")

## Section 2 — Building the Synthetic Spam Dataset

We create **2,000 emails** described by four features:

| Feature | Description | Spam signal |
|---------|-------------|-------------|
| `email_length` | Number of words in the email | Spam emails often very short or very long |
| `num_links` | Hyperlinks in the email | Spam loads up on links |
| `num_caps_words` | ALL-CAPS words (e.g. FREE, WIN) | Classic spam pattern |
| `has_attachment` | Binary flag (0 or 1) | Malicious attachments common in spam |

**Class distribution:** 20% spam (400 emails), 80% not-spam (1,600 emails) — realistic imbalance.

In [ ]:
# ── Cell 2: Generate synthetic spam dataset ───────────────────────────────────
N = 2000
spam_frac = 0.20          # 20 % spam — realistic class imbalance
n_spam = int(N * spam_frac)
n_ham  = N - n_spam

# ── Not-spam (ham) emails ────────────────────────────────────────────────────
ham_length     = np.random.normal(loc=150, scale=60,  size=n_ham).clip(10, 600)
ham_links      = np.random.poisson(lam=1.5,            size=n_ham)
ham_caps       = np.random.poisson(lam=0.5,            size=n_ham)
ham_attachment = np.random.binomial(n=1, p=0.10,       size=n_ham)

# ── Spam emails ──────────────────────────────────────────────────────────────
spam_length     = np.random.normal(loc=80,  scale=40,  size=n_spam).clip(5, 300)
spam_links      = np.random.poisson(lam=6.0,            size=n_spam)
spam_caps       = np.random.poisson(lam=4.0,            size=n_spam)
spam_attachment = np.random.binomial(n=1, p=0.45,       size=n_spam)

# ── Combine into a DataFrame ──────────────────────────────────────────────────
df = pd.DataFrame({
    'email_length'  : np.concatenate([ham_length,     spam_length]),
    'num_links'     : np.concatenate([ham_links,      spam_links]),
    'num_caps_words': np.concatenate([ham_caps,       spam_caps]),
    'has_attachment': np.concatenate([ham_attachment, spam_attachment]),
    'label'         : np.concatenate([np.zeros(n_ham), np.ones(n_spam)])
}).sample(frac=1, random_state=SEED).reset_index(drop=True)  # shuffle

print(f"Dataset shape: {df.shape}")
print(f"Class distribution:\n{df['label'].value_counts().rename({0:'Not-Spam', 1:'Spam'})}")
df.head()

## Section 3 — Training a Logistic Regression to Get Probabilities

We fit a Logistic Regression (covered in depth in Notebook 2) here **solely to obtain probabilities** P(spam|features).  
Think of the model as the judge computing its internal confidence score.

In [ ]:
# ── Cell 3: Train/test split, scale, fit model ────────────────────────────────
FEATURES = ['email_length', 'num_links', 'num_caps_words', 'has_attachment']
X = df[FEATURES].values
y = df['label'].values

# 80/20 stratified split — preserves class ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

# Standardise features (Logistic Regression is gradient-based; scale matters)
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)       # use training stats on test set

# Fit logistic regression
clf = LogisticRegression(random_state=SEED, max_iter=1000)
clf.fit(X_train, y_train)

# Predicted probabilities (column 1 = P(spam))
y_prob = clf.predict_proba(X_test)[:, 1]

print(f"Training samples : {X_train.shape[0]}")
print(f"Test samples     : {X_test.shape[0]}")
print(f"Prob range       : [{y_prob.min():.3f}, {y_prob.max():.3f}]")

## Section 4 — Visualisation 1: Probability Distributions

### What to look for

- Ideally the spam distribution peaks near 1 and the ham distribution peaks near 0.
- The **overlap region** is where the model is uncertain — emails in that zone will flip class as you change the threshold.
- A good model produces **well-separated** histograms.

In [ ]:
# ── Cell 4: Histogram — P(spam) for spam vs. ham emails ──────────────────────
fig, ax = plt.subplots(figsize=(9, 4))

# Separate probabilities by true label
prob_spam = y_prob[y_test == 1]
prob_ham  = y_prob[y_test == 0]

bins = np.linspace(0, 1, 35)

ax.hist(prob_ham,  bins=bins, alpha=0.65, color='steelblue',
        label=f'Not-Spam (n={len(prob_ham)})', edgecolor='white')
ax.hist(prob_spam, bins=bins, alpha=0.65, color='tomato',
        label=f'Spam     (n={len(prob_spam)})', edgecolor='white')

# Mark the default decision threshold
ax.axvline(0.5, color='black', linestyle='--', linewidth=1.8, label='Threshold = 0.5')
ax.fill_betweenx([0, ax.get_ylim()[1] if ax.get_ylim()[1] else 60],
                 0.35, 0.65, alpha=0.12, color='grey', label='Uncertainty zone')

ax.set_xlabel('P(spam | features)', fontsize=12)
ax.set_ylabel('Number of Emails', fontsize=12)
ax.set_title('Model Confidence: How Sure Is the Spam Filter?', fontsize=13, fontweight='bold')
ax.legend(framealpha=0.9)
plt.tight_layout()
plt.show()

print("Interpretation: Emails on the right are confidently spam; on the left, confidently ham.")
print("The grey zone marks where the judge is uncertain — threshold placement matters here.")

## Section 5 — Decision Threshold: The Dial You Can Turn

### Precision vs. Recall Trade-off (Plain English)

| Metric | Question it answers | Formula |
|--------|--------------------|---------|
| **Precision** | Of all emails flagged as spam, how many truly are spam? | TP / (TP + FP) |
| **Recall** | Of all actual spam emails, how many did we catch? | TP / (TP + FN) |
| **F1** | Harmonic mean of precision and recall (balanced metric) | 2·P·R / (P+R) |

**TP** = True Positive (spam correctly flagged)  
**FP** = False Positive (ham wrongly flagged as spam — **annoying**)  
**FN** = False Negative (spam that slipped into your inbox — **dangerous**)

Increasing the threshold → fewer things get called spam → higher precision, lower recall.  
Decreasing the threshold → more things get called spam → higher recall, lower precision.

In [ ]:
# ── Cell 5: Threshold sweep 0.1 → 0.9 ─────────────────────────────────────────
thresholds  = np.arange(0.05, 0.96, 0.01)
precisions  = []
recalls     = []
f1_scores   = []

for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    # zero_division=0 handles the case where no emails are predicted positive
    precisions.append(precision_score(y_test, y_pred_t, zero_division=0))
    recalls.append(   recall_score(   y_test, y_pred_t, zero_division=0))
    f1_scores.append( f1_score(       y_test, y_pred_t, zero_division=0))

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4.5))

ax.plot(thresholds, precisions, color='royalblue',  lw=2.2, label='Precision')
ax.plot(thresholds, recalls,    color='tomato',     lw=2.2, label='Recall')
ax.plot(thresholds, f1_scores,  color='seagreen',   lw=2.2, label='F1 Score')

# Mark the best F1 threshold
best_t = thresholds[np.argmax(f1_scores)]
ax.axvline(best_t, color='seagreen', linestyle=':', lw=1.8,
           label=f'Best F1 threshold ≈ {best_t:.2f}')
ax.axvline(0.5,    color='black',    linestyle='--', lw=1.4, label='Default threshold = 0.5')

ax.set_xlabel('Decision Threshold', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Precision, Recall, and F1 Across Thresholds', fontsize=13, fontweight='bold')
ax.set_xlim(0.05, 0.95)
ax.set_ylim(-0.02, 1.05)
ax.legend(framealpha=0.9)
plt.tight_layout()
plt.show()

print(f"Best F1 achieved at threshold ≈ {best_t:.2f}")
print("Notice: precision and recall are mirror-images of each other as threshold changes.")

## Section 6 — Confusion Matrix at Threshold = 0.5

The confusion matrix is the **ground truth** of classifier evaluation.  
Each cell counts how many emails fell into which category:

```
                 Predicted: Ham    Predicted: Spam
Actual: Ham   [ True Neg (TN)      False Pos (FP) ]
Actual: Spam  [ False Neg (FN)     True Pos  (TP) ]
```

- **FP** = spam filter flags your boss's email as spam ← annoying
- **FN** = a phishing email lands in your inbox ← dangerous

In [ ]:
# ── Cell 6: Confusion matrix heatmap at threshold = 0.5 ───────────────────────
THRESHOLD = 0.5
y_pred_05  = (y_prob >= THRESHOLD).astype(int)
cm         = confusion_matrix(y_test, y_pred_05)

# Compute derived metrics for annotation
tn, fp, fn, tp = cm.ravel()
prec  = tp / (tp + fp) if (tp + fp) > 0 else 0
rec   = tp / (tp + fn) if (tp + fn) > 0 else 0
f1    = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
acc   = (tn + tp) / cm.sum()

# ── Heatmap ───────────────────────────────────────────────────────────────────
labels = np.array([[f'TN = {tn}\n(Ham → Inbox ✓)',
                    f'FP = {fp}\n(Ham → Spam ✗)'],
                   [f'FN = {fn}\n(Spam → Inbox ✗)',
                    f'TP = {tp}\n(Spam → Spam ✓)']])

fig, ax = plt.subplots(figsize=(6.5, 5))
sns.heatmap(cm, annot=labels, fmt='', cmap='Blues', linewidths=1.5,
            xticklabels=['Predicted: Ham', 'Predicted: Spam'],
            yticklabels=['Actual: Ham',    'Actual: Spam'], ax=ax,
            annot_kws={'size': 12})
ax.set_title(f'Confusion Matrix  (threshold = {THRESHOLD})', fontsize=13, fontweight='bold')
ax.set_ylabel('True Label', fontsize=12)
ax.set_xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

print(f"Accuracy  = {acc:.3f}")
print(f"Precision = {prec:.3f}  (of emails flagged as spam, {prec*100:.1f}% are truly spam)")
print(f"Recall    = {rec:.3f}  (we caught {rec*100:.1f}% of all actual spam emails)")
print(f"F1 Score  = {f1:.3f}")

## Section 7 — ROC Curve and AUC

### Plain English

The **ROC curve** (Receiver Operating Characteristic) shows the trade-off between:
- **True Positive Rate (Recall):** how many spam emails we catch
- **False Positive Rate:** how many ham emails we wrongly flag

Each point on the curve is one threshold value.  
**AUC** (Area Under the Curve) summarises the entire curve as a single number:
- AUC = 1.0 → perfect classifier
- AUC = 0.5 → random guessing (the diagonal dashed line)
- AUC = 0.7 → decent; AUC ≥ 0.9 → very strong

**AUC interpretation:** the probability that the model ranks a randomly chosen spam email higher than a randomly chosen ham email.

In [ ]:
# ── Cell 7: ROC curve with AUC shaded ─────────────────────────────────────────
fpr, tpr, roc_thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(7, 6))

# Shade the area under the ROC curve
ax.fill_between(fpr, tpr, alpha=0.18, color='royalblue', label=f'AUC = {roc_auc:.3f}')
ax.plot(fpr, tpr, color='royalblue', lw=2.5, label='ROC Curve')

# Random baseline
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Guess (AUC = 0.5)')

# Mark the operating point at threshold = 0.5
idx_05 = np.argmin(np.abs(roc_thresholds - 0.5))
ax.scatter(fpr[idx_05], tpr[idx_05], s=120, color='tomato', zorder=5,
           label=f'Threshold = 0.5 → FPR={fpr[idx_05]:.2f}, TPR={tpr[idx_05]:.2f}')

ax.set_xlabel('False Positive Rate  (Ham wrongly flagged as Spam)', fontsize=12)
ax.set_ylabel('True Positive Rate  (Spam correctly caught)', fontsize=12)
ax.set_title('ROC Curve — Spam Email Classifier', fontsize=13, fontweight='bold')
ax.legend(framealpha=0.9, loc='lower right')
ax.set_xlim(-0.01, 1.01)
ax.set_ylim(-0.01, 1.01)
plt.tight_layout()
plt.show()

print(f"AUC = {roc_auc:.4f}")
print("AUC close to 1.0 means the model's confidence scores are well-separated between classes.")

## Section 8 — The Decision Boundary in 2D Feature Space

### What is a Decision Boundary?

> The **decision boundary** is the set of all points in feature space where P(y=1|x) = threshold.  
> On one side the model predicts spam; on the other, ham.

For Logistic Regression with two features, the decision boundary is a **straight line** (linear classifier).  
We project our data onto `email_length` and `num_links` to visualise this in 2D.

**Contourf** colours the entire feature space: red = predicted spam region, blue = predicted ham region.

In [ ]:
# ── Cell 8: 2D decision boundary using 2 features ─────────────────────────────
FEAT2 = ['email_length', 'num_links']
X2 = df[FEAT2].values
y2 = df['label'].values

# Re-scale and refit on 2-feature data
scaler2 = StandardScaler()
X2_sc   = scaler2.fit_transform(X2)
clf2    = LogisticRegression(random_state=SEED, max_iter=1000)
clf2.fit(X2_sc, y2)

# ── Build a mesh grid over the feature space ──────────────────────────────────
x_min, x_max = X2[:, 0].min() - 20,  X2[:, 0].max() + 20
y_min, y_max = X2[:, 1].min() - 1,   X2[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 400),
                     np.linspace(y_min, y_max, 400))

# Predict probability across the grid
grid_scaled = scaler2.transform(np.c_[xx.ravel(), yy.ravel()])
Z = clf2.predict_proba(grid_scaled)[:, 1].reshape(xx.shape)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
cf = ax.contourf(xx, yy, Z, levels=50, cmap='RdBu_r', alpha=0.75)
plt.colorbar(cf, ax=ax, label='P(spam)')

# Decision boundary contour at P = 0.5
ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2.5, linestyles='--')

# Scatter actual data (subsample for clarity)
idx_ham  = np.where(y2 == 0)[0][:300]
idx_spam = np.where(y2 == 1)[0][:300]
ax.scatter(X2[idx_ham,  0], X2[idx_ham,  1], s=18, color='steelblue', alpha=0.5, label='Ham')
ax.scatter(X2[idx_spam, 0], X2[idx_spam, 1], s=18, color='tomato',    alpha=0.6, label='Spam')

ax.set_xlabel('Email Length (words)', fontsize=12)
ax.set_ylabel('Number of Links', fontsize=12)
ax.set_title('Decision Boundary: P(spam) = 0.5  (black dashed line)', fontsize=13, fontweight='bold')
ax.legend(framealpha=0.9)
plt.tight_layout()
plt.show()

print("The dashed black line is the decision boundary — Logistic Regression always draws a straight line.")

## Section 9 — The Accuracy Paradox

### The Dangerous Dummy Classifier

Suppose our dataset has 80% ham and 20% spam.  
A **dummy classifier** that *always predicts ham* achieves:

> **Accuracy = 80%** — and catches **zero spam emails**.

This is the **accuracy paradox**: high accuracy on imbalanced datasets is meaningless if it comes from always predicting the majority class.

In a medical context: if 99% of patients are healthy, a model saying "everyone is healthy" hits 99% accuracy but **misses every cancer patient**.

In [ ]:
# ── Cell 9: Accuracy paradox demonstration ────────────────────────────────────
# Dummy: always predict NOT spam (class 0)
y_dummy = np.zeros_like(y_test)

dummy_acc  = accuracy_score(y_test, y_dummy)
dummy_prec = precision_score(y_test, y_dummy, zero_division=0)
dummy_rec  = recall_score(   y_test, y_dummy, zero_division=0)
dummy_f1   = f1_score(       y_test, y_dummy, zero_division=0)

# Real model at threshold = 0.5
real_acc   = accuracy_score(y_test, y_pred_05)
real_prec  = precision_score(y_test, y_pred_05, zero_division=0)
real_rec   = recall_score(   y_test, y_pred_05, zero_division=0)
real_f1    = f1_score(       y_test, y_pred_05, zero_division=0)

# ── Summary table ─────────────────────────────────────────────────────────────
comparison = pd.DataFrame({
    'Accuracy'  : [dummy_acc,  real_acc],
    'Precision' : [dummy_prec, real_prec],
    'Recall'    : [dummy_rec,  real_rec],
    'F1 Score'  : [dummy_f1,   real_f1]
}, index=['Always-Ham (Dummy)', 'Logistic Regression']).round(3)

print("=" * 60)
print("THE ACCURACY PARADOX — Imbalanced Class Evaluation")
print("=" * 60)
print(comparison.to_string())
print()
print(f"The dummy classifier gets {dummy_acc*100:.1f}% accuracy but catches 0% of spam!")
print(f"Logistic Regression: {real_acc*100:.1f}% accuracy but catches {real_rec*100:.1f}% of spam.")
print()
print("Lesson: ALWAYS report precision, recall, and F1 — not just accuracy — for imbalanced problems.")

In [ ]:
# ── Cell 10: Bar chart comparing dummy vs. real model ─────────────────────────
metrics      = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
dummy_scores = [dummy_acc, dummy_prec, dummy_rec, dummy_f1]
real_scores  = [real_acc,  real_prec,  real_rec,  real_f1]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, dummy_scores, width, label='Always-Ham (Dummy)',
               color='#d9534f', alpha=0.85, edgecolor='white')
bars2 = ax.bar(x + width/2, real_scores,  width, label='Logistic Regression',
               color='#5bc0de', alpha=0.85, edgecolor='white')

# Value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=10)

ax.set_ylabel('Score', fontsize=12)
ax.set_title('The Accuracy Paradox — Why F1 Matters More Than Accuracy', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=12)
ax.set_ylim(0, 1.15)
ax.legend(framealpha=0.9)
plt.tight_layout()
plt.show()

## Section 10 — Feature Distributions by Class

Before wrapping up, let us look at what makes spam emails different.  
These distributions confirm the signals we baked into the data generation.

In [ ]:
# ── Cell 11: Feature distributions by class ───────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
feature_labels = {
    'email_length'  : 'Email Length (words)',
    'num_links'     : 'Number of Links',
    'num_caps_words': 'ALL-CAPS Words',
    'has_attachment': 'Has Attachment (0/1)'
}

for ax, (feat, label) in zip(axes.flat, feature_labels.items()):
    ham_vals  = df.loc[df['label'] == 0, feat]
    spam_vals = df.loc[df['label'] == 1, feat]

    if feat == 'has_attachment':
        # Bar chart for binary feature
        vals_ham  = ham_vals.value_counts(normalize=True).sort_index()
        vals_spam = spam_vals.value_counts(normalize=True).sort_index()
        x_pos = np.array([0, 1])
        ax.bar(x_pos - 0.2, vals_ham.reindex([0,1], fill_value=0),  0.35,
               label='Ham',  color='steelblue', alpha=0.8)
        ax.bar(x_pos + 0.2, vals_spam.reindex([0,1], fill_value=0), 0.35,
               label='Spam', color='tomato',    alpha=0.8)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['No', 'Yes'])
        ax.set_ylabel('Proportion')
    else:
        ax.hist(ham_vals,  bins=30, alpha=0.6, color='steelblue', label='Ham',  edgecolor='white')
        ax.hist(spam_vals, bins=30, alpha=0.6, color='tomato',    label='Spam', edgecolor='white')

    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)

fig.suptitle('Feature Distributions: Spam vs. Ham Emails', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Section 11 — Concept Summary

| Concept | Key Point |
|---------|----------|
| **Model Output** | Always a probability P(y=1\|x) ∈ [0,1] — not a hard label |
| **Threshold** | Converts probability → class label; default 0.5 is not always best |
| **Decision Boundary** | Where P(y=1\|x) = threshold; linear for Logistic Regression |
| **Precision** | Of all spam predictions, how many are correct? |
| **Recall** | Of all actual spam, how many did we catch? |
| **F1** | Balanced metric; use when classes are imbalanced |
| **AUC** | Threshold-independent quality metric; 1.0 = perfect |
| **Accuracy Paradox** | 80% accuracy on 80/20 dataset is meaningless if you always predict majority class |

### When to Lower the Threshold (increase recall)
- Cancer screening, fraud detection, disease diagnosis
- Missing a positive is **more costly** than a false alarm

### When to Raise the Threshold (increase precision)
- Email spam filters (you don't want legitimate emails flagged)
- Legal/HR systems where a false accusation is very costly

---

## Self-Check Questions

Test your understanding. Try to answer each question before revealing the answer.

---

**Question 1:**  
A spam filter has AUC = 0.97 but precision = 0.20.  
What does precision = 0.20 mean practically?

<details>
<summary>Show Answer</summary>

**Precision = 0.20** means that for every 5 emails the filter flags as spam, only **1 is actually spam** — the other 4 are legitimate emails wrongly quarantined (False Positives).

AUC = 0.97 tells us the model's **probability rankings** are excellent (it ranks spam emails higher than ham emails 97% of the time). But because the threshold is set too low (or the class imbalance is severe), many ham emails score above the threshold too, tanking precision.

**Practical impact:** Users will find their boss's emails, newsletters, and receipts sitting in the Spam folder. They'll lose trust in the filter and start ignoring it — or disable it entirely.

**Fix:** Raise the decision threshold, or adjust the class weights so the model is more conservative about flagging spam.
</details>

---

**Question 2:**  
You are building a cancer screening tool.  
Should you raise or lower the threshold from 0.5 to maximise recall? Why?

<details>
<summary>Show Answer</summary>

You should **lower** the threshold (e.g. to 0.3 or even 0.2).

**Why:**
- Recall measures: "Of all actual cancer cases, how many did we catch?"
- Lowering the threshold makes the model more trigger-happy — it flags more cases as positive.
- More flagged cases → more actual cancer patients detected → **higher recall**.
- The trade-off is lower precision: more healthy patients will be sent for unnecessary follow-up biopsies (False Positives). This is unpleasant and costly, but much better than **missing a cancer patient** (False Negative = missing cancer = potentially fatal).

**Rule of thumb:** When a False Negative (missed positive) is much more costly than a False Positive (false alarm), lower your threshold.
</details>

---

**Question 3:**  
A logistic regression model is trained on 10 features.  
What shape is its decision boundary in 10-dimensional feature space?

<details>
<summary>Show Answer</summary>

The decision boundary is a **hyperplane** — the 10-dimensional generalisation of a straight line.

For logistic regression, the decision boundary is defined by:
$$\theta_0 + \theta_1 x_1 + \theta_2 x_2 + \cdots + \theta_{10} x_{10} = 0$$

This is always a **linear equation**, making logistic regression a **linear classifier**.

- In 2D: a **line**
- In 3D: a **plane**
- In 10D: a **hyperplane**

**Implication:** Logistic regression cannot learn curved or non-linear decision boundaries without feature engineering (e.g. adding polynomial features like x₁² or x₁·x₂). For naturally non-linear problems, you'd need Decision Trees, SVMs with kernel, or Neural Networks.
</details>